`category_topic_merge_mapping_notebook.py`는 기존 세부 `topic`을 더 큰 `merged_topic`으로 통합하는 Databricks 노트북입니다.

- `topic_pool`을 카테고리·감성별로 읽어옵니다.
- LLM으로 유사 topic을 통합 매핑합니다.
- 단순 전반 긍/부정은 각각 `전반적 긍정`, `전반적 부정`으로 고정합니다.
- 매핑 결과를 detail/summary 테이블에 적용합니다.


`category_topic_merge_mapping_notebook.py`는 기존 세부 `topic`을 더 큰 `merged_topic`으로 통합하는 Databricks 노트북입니다.

- `topic_pool`을 카테고리·감성별로 읽어옵니다.
- LLM으로 유사 topic을 통합 매핑합니다.
- 단순 전반 긍/부정은 각각 `전반적 긍정`, `전반적 부정`으로 고정합니다.
- 매핑 결과를 detail/summary 테이블에 적용합니다.


In [0]:
# Databricks notebook source
# COMMAND ----------
# 1) Config

from __future__ import annotations

import json
import re
import time
from typing import Any, Dict, List, Optional, Tuple

import pandas as pd
from pyspark.sql import DataFrame, Window
from pyspark.sql import functions as F
from pyspark.sql import types as T

ENDPOINT = "databricks-gpt-5-4"
SAVE_DB = "sandbox.z_jungryo_lee"

# 사용자가 현재 확인 중인 topic_pool 버전에 맞춰 바꿔주세요.
SOURCE_PROMPT_VERSION = "category_topic_dynamic_rules_allgroups_v1"
SOURCE_TOPIC_POOL_TABLE = f"{SAVE_DB}.category_topic_pool_{SOURCE_PROMPT_VERSION}"
SOURCE_DETAIL_TABLE = f"{SAVE_DB}.category_topic_detail_{SOURCE_PROMPT_VERSION}"

MERGE_PROMPT_VERSION = f"{SOURCE_PROMPT_VERSION}_merge_v2"
TOPIC_MERGE_MAPPING_TABLE = f"{SAVE_DB}.category_topic_catalog_v2"
MERGED_DETAIL_TABLE = f"{SAVE_DB}.category_topic_detail_merged_{MERGE_PROMPT_VERSION}"
MERGED_SUMMARY_TABLE = f"{SAVE_DB}.category_topic_summary_merged_{MERGE_PROMPT_VERSION}"
FAILED_MERGE_TABLE = f"{SAVE_DB}.category_topic_merge_failed_{MERGE_PROMPT_VERSION}"

LOW_SHARE_PCT = 1.0  # 리뷰 비율 1% 미만이면서 어느 그룹에도 안 맞으면 "기타"
MIN_ROWS_PER_TOPIC = 10
MAX_TOKENS = 5000
MAX_RETRIES = 3
BACKOFF_BASE = 1.8
RATE_LIMIT_SECONDS = 0.4

print(f"[CONFIG] source_topic_pool={SOURCE_TOPIC_POOL_TABLE}")
print(f"[CONFIG] output_mapping={TOPIC_MERGE_MAPPING_TABLE}")
print(f"[CONFIG] LOW_SHARE_PCT={LOW_SHARE_PCT}%")


# COMMAND ----------
# 2) Helpers  

def clean_text(x: Any) -> str:
    return "" if x is None else re.sub(r"\s+", " ", str(x)).strip()


def extract_json(text: str) -> Dict[str, Any]:
    text = clean_text(text)
    try:
        return json.loads(text)
    except Exception:
        pass

    match = re.search(r"\{.*\}", text, flags=re.S)
    if not match:
        raise ValueError(f"Invalid JSON: {text[:1000]}")

    candidate = match.group(0)
    candidate = re.sub(r",\s*}", "}", candidate)
    candidate = re.sub(r",\s*]", "]", candidate)
    return json.loads(candidate)


def call_llm(messages: List[Dict[str, str]], max_tokens: int = MAX_TOKENS) -> Dict[str, Any]:
    from mlflow.deployments import get_deploy_client

    client = get_deploy_client("databricks")
    payload = {
        "messages": messages,
        "temperature": 0.0,
        "max_tokens": max_tokens,
    }

    last_error: Optional[Exception] = None
    for attempt in range(MAX_RETRIES):
        resp = None
        try:
            print(f"[LLM CALL] attempt={attempt + 1}/{MAX_RETRIES}")
            resp = client.predict(endpoint=ENDPOINT, inputs=payload)

            if isinstance(resp, dict):
                if "choices" in resp and resp["choices"]:
                    return extract_json(resp["choices"][0]["message"]["content"])
                if "predictions" in resp and resp["predictions"]:
                    pred0 = resp["predictions"][0]
                    if isinstance(pred0, dict) and "content" in pred0:
                        return extract_json(pred0["content"])
                    if isinstance(pred0, str):
                        return extract_json(pred0)
                if "content" in resp:
                    return extract_json(resp["content"])

            if isinstance(resp, str):
                return extract_json(resp)

            raise ValueError(f"Unexpected response schema: {resp}")
        except Exception as exc:
            last_error = exc
            print(f"[LLM ERROR] attempt={attempt + 1}/{MAX_RETRIES}, error={repr(exc)}")
            if resp is not None:
                print(str(resp)[:2000])
            time.sleep(BACKOFF_BASE ** attempt)

    raise RuntimeError(f"LLM call failed: {repr(last_error)}")


def save_table(df: DataFrame, table_name: str) -> None:
    if spark.catalog.tableExists(table_name):
        print(f"[SAVE] overwrite -> {table_name}")
    else:
        print(f"[SAVE] create -> {table_name}")
    df.write.mode("overwrite").format("delta").saveAsTable(table_name)


def append_table(df: DataFrame, table_name: str) -> None:
    if spark.catalog.tableExists(table_name):
        print(f"[SAVE] append -> {table_name}")
        df.write.mode("append").format("delta").saveAsTable(table_name)
    else:
        print(f"[SAVE] create -> {table_name}")
        df.write.mode("overwrite").format("delta").saveAsTable(table_name)


def delete_group_rows(table_name: str, cate_1_depth: str, cate_2_depth: str, sc_measurement: int) -> None:
    if not spark.catalog.tableExists(table_name):
        return
    safe_cate_1 = cate_1_depth.replace("'", "''")
    safe_cate_2 = cate_2_depth.replace("'", "''")
    spark.sql(
        f"""
        delete from {table_name}
        where cate_1_depth = '{safe_cate_1}'
          and cate_2_depth = '{safe_cate_2}'
          and sc_measurement = {int(sc_measurement)}
        """
    )


def sc_label(sc_measurement: int) -> str:
    if int(sc_measurement) == 1:
        return "긍정"
    if int(sc_measurement) == -1:
        return "부정"
    return "중립"


OVERALL_POSITIVE_TOPIC = "전반적 긍정"
OVERALL_NEGATIVE_TOPIC = "전반적 부정"
OVERALL_LOCKED_TOPICS = [OVERALL_POSITIVE_TOPIC, OVERALL_NEGATIVE_TOPIC]


def forced_overall_topic(source_topic: Any, source_description: Any, sc_measurement: int) -> Optional[str]:
    """단순 전반 긍/부정 topic은 카테고리와 무관하게 고정 그룹으로 보낸다."""
    topic = clean_text(source_topic)
    description = clean_text(source_description)
    text = f"{topic} {description}"
    sc = int(sc_measurement)

    if sc == 1:
        sentiment_markers = ("긍정", "만족", "호평", "감탄")
        target_topic = OVERALL_POSITIVE_TOPIC
    elif sc == -1:
        sentiment_markers = ("부정", "불만", "불만족")
        target_topic = OVERALL_NEGATIVE_TOPIC
    else:
        return None

    if target_topic in topic:
        return target_topic

    simple_overall_markers = (
        "전반",
        "전반적",
        "전반적으로",
        "막연",
        "구체 사유 없이",
        "구체 이유 없이",
        "구체 근거 없이",
        "이유 불명",
        "단순",
        "짧게",
        "초단문",
        "전반 만족",
        "전반 호평",
    )

    has_sentiment = any(marker in topic for marker in sentiment_markers)
    has_simple_overall_context = any(marker in text for marker in simple_overall_markers)
    if has_sentiment and has_simple_overall_context:
        return target_topic

    return None


# COMMAND ----------
# 3) Load Topic Pool & Review Share

topic_pool_df = (
    spark.table(SOURCE_TOPIC_POOL_TABLE)
    .select(
        "cate_1_depth",
        "cate_2_depth",
        F.col("sc_measurement").cast("int").alias("sc_measurement"),
        F.col("topic_order").cast("int").alias("topic_order"),
        "topic",
        "description",
        "representative_memos_json",
    )
    .where(F.col("sc_measurement").isin(-1, 1))
)

# detail 테이블에서 topic별 리뷰 건수/비율 계산
if spark.catalog.tableExists(SOURCE_DETAIL_TABLE):
    detail_counts_df = (
        spark.table(SOURCE_DETAIL_TABLE)
        .where(F.col("sc_measurement").cast("int").isin(-1, 1))
        .groupBy("cate_1_depth", "cate_2_depth", F.col("sc_measurement").cast("int").alias("sc_measurement"), "topic")
        .agg(F.count("*").alias("review_count"))
    )
    detail_totals_df = (
        detail_counts_df
        .groupBy("cate_1_depth", "cate_2_depth", "sc_measurement")
        .agg(F.sum("review_count").alias("group_total"))
    )
    topic_share_df = (
        detail_counts_df.alias("c")
        .join(
            detail_totals_df.alias("t"),
            on=["cate_1_depth", "cate_2_depth", "sc_measurement"],
            how="inner",
        )
        .withColumn("review_share_pct", F.round(F.col("review_count") / F.col("group_total") * 100, 2))
        .select("cate_1_depth", "cate_2_depth", "sc_measurement", "topic", "review_count", "review_share_pct")
    )

    topic_pool_df = (
        topic_pool_df.alias("p")
        .join(
            topic_share_df.alias("s"),
            on=[
                F.col("p.cate_1_depth") == F.col("s.cate_1_depth"),
                F.col("p.cate_2_depth") == F.col("s.cate_2_depth"),
                F.col("p.sc_measurement") == F.col("s.sc_measurement"),
                F.col("p.topic") == F.col("s.topic"),
            ],
            how="left",
        )
        .select(
            F.col("p.cate_1_depth"),
            F.col("p.cate_2_depth"),
            F.col("p.sc_measurement"),
            F.col("p.topic_order"),
            F.col("p.topic"),
            F.col("p.description"),
            F.col("p.representative_memos_json"),
            F.coalesce(F.col("s.review_count"), F.lit(0)).alias("review_count"),
            F.coalesce(F.col("s.review_share_pct"), F.lit(0.0)).alias("review_share_pct"),
        )
    )
    print("[INFO] review_share_pct calculated from detail table")
else:
    topic_pool_df = topic_pool_df.withColumn("review_count", F.lit(0)).withColumn("review_share_pct", F.lit(0.0))
    print(f"[WARN] detail table not found, review_share_pct = 0: {SOURCE_DETAIL_TABLE}")

display(topic_pool_df.orderBy("cate_1_depth", "cate_2_depth", "sc_measurement", "topic_order"))


# COMMAND ----------
# 4) Prompt

def build_merge_messages(group: Dict[str, Any], topics: List[Dict[str, Any]], max_groups: int) -> List[Dict[str, str]]:
    sentiment = sc_label(int(group["sc_measurement"]))
    overall = "전반적 긍정" if int(group["sc_measurement"]) == 1 else "전반적 부정"
    topic_count = len(topics)

    system = f"""
너는 VOC 토픽 택소노미를 정리하는 분석가다.
입력은 이미 만들어진 세부 topic {topic_count}개 목록이며, 각 topic에는 review_share_pct(해당 그룹 내 리뷰 비율 %)가 포함되어 있다.
topic/description/대표문장/리뷰비율을 보고 "큰 단위" 통합 매핑표를 만든다.

━━ 핵심 원칙: 넓게 묶기 ━━
이 작업의 목적은 세부 topic을 "분석에 유용한 대주제"로 통합하는 것이다.
통합그룹은 기존 세부 topic보다 확연히 적어야 하며, 하나의 통합그룹에 여러 개의 세부 topic이 포함되어야 한다.

넓게 묶는 기준:
- 의미적 유사성: 같은 제품 속성/기능/경험을 다루는 topic들은 하나로 묶는다.
- 분석 관점: 의사결정자가 "함께 보면 의미 있는" topic들을 묶는다. 예) 화질+색감+밝기 → "화면 품질", 설치+배송+무게 → "설치/배송 경험"
- 상위 개념 포함: 관련된 세부 속성들을 상위 개념으로 묶는다. 예) 크기+두께+무게 → "외형/사이즈", 소음+진동+발열 → "동작 안정성"

━━ 통합그룹 수 제한 ━━
- 통합그룹 수는 최대 {max_groups}개 이하로 만든다 (세부 topic {topic_count}개의 절반 미만).
- 적을수록 좋다. 가능한 한 많은 세부 topic을 하나의 통합그룹으로 묶어라.

━━ "기타" 그룹 규칙 ━━
- review_share_pct < {LOW_SHARE_PCT}% 인 topic이 다른 통합그룹에 의미적으로 맞지 않으면 "기타" 통합그룹에 넣는다.
- 단, review_share_pct < {LOW_SHARE_PCT}%라도 다른 topic과 의미적으로 명확히 관련되면 해당 통합그룹에 포함한다.
- "기타" 그룹은 "의미적으로 어디에도 속하지 않는 잡다한 소수 topic" 집합이다.

━━ "전반적 {sentiment}" 고정 그룹 ━━
- 단순 감탄/불만처럼 구체 이유가 없는 topic만 포함한다.
- 구체 속성(크기/색감/기능/AS/설치 등)이 있는 topic은 절대 넣지 않는다.
- 카테고리별 단순 긍정/부정 또는 전반적 긍정/부정 topic은 각각 "전반적 긍정", "전반적 부정"으로만 매핑한다.

━━ 기타 규칙 ━━
- 같은 polarity({sentiment})와 같은 cate_1/cate_2 안에서만 묶는다.
- 통합그룹명은 짧고 분석 가능한 한국어 명사구로 작성한다.
- 기존 topic명은 입력 그대로 source_topic에 복사한다.
- 출력은 반드시 기존 topic 1개당 매핑 row 1개를 포함해야 한다.

JSON만 출력한다:
{{
  "rows": [
    {{
      "source_topic": "기존 topic명",
      "merged_topic": "통합그룹명",
      "merged_description": "통합그룹 설명",
      "merge_reason": "왜 이 통합그룹에 포함했는지 짧게"
    }}
  ]
}}
"""

    # topic 정보에 review_share_pct 포함
    topics_for_prompt = []
    for t in topics:
        topics_for_prompt.append({
            "topic_order": t.get("topic_order"),
            "topic": t.get("topic"),
            "description": t.get("description"),
            "review_share_pct": t.get("review_share_pct", 0.0),
            "representative_memos_json": t.get("representative_memos_json"),
        })

    user = {
        "cate_1_depth": group["cate_1_depth"],
        "cate_2_depth": group["cate_2_depth"],
        "sc_measurement": int(group["sc_measurement"]),
        "sentiment": sentiment,
        "topic_count": topic_count,
        "max_merged_groups": max_groups,
        "topics": topics_for_prompt,
    }

    return [
        {"role": "system", "content": clean_text(system)},
        {"role": "user", "content": json.dumps(user, ensure_ascii=False)},
    ]


# COMMAND ----------
# 5) Build Merge Mapping Table

mapping_schema = T.StructType([
    T.StructField("cate_1_depth", T.StringType(), True),
    T.StructField("cate_2_depth", T.StringType(), True),
    T.StructField("sc_measurement", T.IntegerType(), True),
    T.StructField("source_topic_order", T.IntegerType(), True),
    T.StructField("source_topic", T.StringType(), True),
    T.StructField("source_description", T.StringType(), True),
    T.StructField("merged_topic", T.StringType(), True),
    T.StructField("merged_description", T.StringType(), True),
    T.StructField("merge_reason", T.StringType(), True),
])

failed_schema = T.StructType([
    T.StructField("cate_1_depth", T.StringType(), True),
    T.StructField("cate_2_depth", T.StringType(), True),
    T.StructField("sc_measurement", T.IntegerType(), True),
    T.StructField("error_message", T.StringType(), True),
    T.StructField("event_ts", T.TimestampType(), True),
])

groups = [
    r.asDict()
    for r in topic_pool_df.select("cate_1_depth", "cate_2_depth", "sc_measurement").distinct()
    .orderBy("cate_1_depth", "cate_2_depth", "sc_measurement")
    .collect()
]

all_mapping_rows: List[Dict[str, Any]] = []

for idx, group in enumerate(groups, start=1):
    print(
        f"[MERGE GROUP] {idx}/{len(groups)} "
        f"{group['cate_1_depth']} | {group['cate_2_depth']} | {group['sc_measurement']}"
    )

    group_topics = [
        r.asDict()
        for r in topic_pool_df
        .where(
            (F.col("cate_1_depth") == group["cate_1_depth"])
            & (F.col("cate_2_depth") == group["cate_2_depth"])
            & (F.col("sc_measurement") == int(group["sc_measurement"]))
        )
        .orderBy("topic_order")
        .collect()
    ]

    source_by_topic = {clean_text(t["topic"]): t for t in group_topics}

    # 동적 max_groups: 세부 topic 수의 절반 미만
    max_groups = max(2, len(group_topics) // 2)
    print(f"  topic_count={len(group_topics)}, max_merged_groups={max_groups}")

    try:
        result = call_llm(build_merge_messages(group, group_topics, max_groups))
        result_rows = result.get("rows", [])
        mapped_source_topics = set()

        for row in result_rows:
            source_topic = clean_text(row.get("source_topic"))
            if source_topic not in source_by_topic:
                continue

            source = source_by_topic[source_topic]
            mapped_source_topics.add(source_topic)
            locked_overall_topic = forced_overall_topic(
                source_topic,
                source["description"],
                int(group["sc_measurement"]),
            )
            merged_topic = locked_overall_topic or clean_text(row.get("merged_topic")) or source_topic
            merged_description = (
                "구체 사유 없이 전반적으로 긍정 평가한 단순 총평"
                if locked_overall_topic == OVERALL_POSITIVE_TOPIC
                else "구체 사유 없이 전반적으로 부정 평가한 단순 총평"
                if locked_overall_topic == OVERALL_NEGATIVE_TOPIC
                else clean_text(row.get("merged_description")) or clean_text(source["description"])
            )
            merge_reason = (
                f"단순 전반 {sc_label(int(group['sc_measurement']))} topic으로 카테고리 공통 고정 그룹 유지"
                if locked_overall_topic
                else clean_text(row.get("merge_reason"))
            )
            all_mapping_rows.append(
                {
                    "cate_1_depth": group["cate_1_depth"],
                    "cate_2_depth": group["cate_2_depth"],
                    "sc_measurement": int(group["sc_measurement"]),
                    "source_topic_order": int(source["topic_order"]),
                    "source_topic": source_topic,
                    "source_description": clean_text(source["description"]),
                    "merged_topic": merged_topic,
                    "merged_description": merged_description,
                    "merge_reason": merge_reason,
                }
            )

        # LLM이 빠뜨린 source topic은 안전하게 자기 자신으로 매핑한다.
        missing_topics = [t for t in group_topics if clean_text(t["topic"]) not in mapped_source_topics]
        for source in missing_topics:
            source_topic = clean_text(source["topic"])
            locked_overall_topic = forced_overall_topic(
                source_topic,
                source["description"],
                int(group["sc_measurement"]),
            )
            all_mapping_rows.append(
                {
                    "cate_1_depth": group["cate_1_depth"],
                    "cate_2_depth": group["cate_2_depth"],
                    "sc_measurement": int(group["sc_measurement"]),
                    "source_topic_order": int(source["topic_order"]),
                    "source_topic": source_topic,
                    "source_description": clean_text(source["description"]),
                    "merged_topic": locked_overall_topic or source_topic,
                    "merged_description": (
                        "구체 사유 없이 전반적으로 긍정 평가한 단순 총평"
                        if locked_overall_topic == OVERALL_POSITIVE_TOPIC
                        else "구체 사유 없이 전반적으로 부정 평가한 단순 총평"
                        if locked_overall_topic == OVERALL_NEGATIVE_TOPIC
                        else clean_text(source["description"])
                    ),
                    "merge_reason": (
                        f"LLM 누락, 단순 전반 {sc_label(int(group['sc_measurement']))} topic으로 카테고리 공통 고정 그룹 유지"
                        if locked_overall_topic
                        else "LLM 누락으로 원 topic 유지"
                    ),
                }
            )

        time.sleep(RATE_LIMIT_SECONDS)
    except Exception as exc:
        print(f"[MERGE FAILED] {repr(exc)}")
        append_table(
            spark.createDataFrame(
                [(
                    group["cate_1_depth"],
                    group["cate_2_depth"],
                    int(group["sc_measurement"]),
                    repr(exc),
                    pd.Timestamp.now().to_pydatetime(),
                )],
                schema=failed_schema,
            ),
            FAILED_MERGE_TABLE,
        )

mapping_df = spark.createDataFrame(pd.DataFrame(all_mapping_rows), schema=mapping_schema)
save_table(mapping_df, TOPIC_MERGE_MAPPING_TABLE)

display(
    mapping_df.orderBy(
        "cate_1_depth",
        "cate_2_depth",
        "sc_measurement",
        "merged_topic",
        "source_topic_order",
    )
)


# COMMAND ----------
# 6) Display As "통합그룹별 포함 기존 Topic" Table

mapping_df = spark.table(TOPIC_MERGE_MAPPING_TABLE)

merged_group_view_df = (
    mapping_df
    .groupBy(
        "cate_1_depth",
        "cate_2_depth",
        "sc_measurement",
        "merged_topic",
        "merged_description",
    )
    .agg(
        F.sort_array(
            F.collect_list(
                F.struct(
                    F.col("source_topic_order").alias("order"),
                    F.col("source_topic").alias("topic"),
                )
            )
        ).alias("source_topics_struct")
    )
    .withColumn(
        "included_existing_topics",
        F.expr("transform(source_topics_struct, x -> concat(cast(x.order as string), '. ', x.topic))")
    )
    .drop("source_topics_struct")
    .orderBy("cate_1_depth", "cate_2_depth", "sc_measurement", "merged_topic")
)

display(merged_group_view_df)


# COMMAND ----------
# 7) Optional - Apply Mapping To Existing Detail And Rebuild Summary
# 이 셀은 기존 detail을 덮어쓰지 않고 merged 전용 테이블을 새로 저장합니다.

if spark.catalog.tableExists(SOURCE_DETAIL_TABLE):
    detail_df = spark.table(SOURCE_DETAIL_TABLE)
    mapping_df = spark.table(TOPIC_MERGE_MAPPING_TABLE)

    merged_detail_base_df = (
        detail_df.alias("d")
        .join(
            mapping_df.alias("m"),
            on=[
                F.col("d.cate_1_depth") == F.col("m.cate_1_depth"),
                F.col("d.cate_2_depth") == F.col("m.cate_2_depth"),
                F.col("d.sc_measurement").cast("int") == F.col("m.sc_measurement"),
                F.col("d.topic") == F.col("m.source_topic"),
            ],
            how="left",
        )
        .select(
            F.col("d._row_id"),
            F.col("d.cate_1_depth"),
            F.col("d.cate_2_depth"),
            F.col("d.sc_measurement").cast("int").alias("sc_measurement"),
            F.col("d.memo"),
            F.col("d.topic").alias("source_topic"),
            F.coalesce(F.col("m.merged_topic"), F.col("d.topic")).alias("merged_topic_raw"),
            F.coalesce(F.col("m.merged_description"), F.col("d.description")).alias("merged_description_raw"),
        )
    )

    merged_counts_df = (
        merged_detail_base_df
        .groupBy("cate_1_depth", "cate_2_depth", "sc_measurement", "merged_topic_raw")
        .agg(F.count("*").alias("merged_topic_count"))
    )

    rare_merged_df = (
        merged_counts_df
        .where(
            (F.col("merged_topic_count") < F.lit(MIN_ROWS_PER_TOPIC))
            & (~F.col("merged_topic_raw").isin(OVERALL_LOCKED_TOPICS))
        )
        .select(
            "cate_1_depth",
            "cate_2_depth",
            "sc_measurement",
            F.col("merged_topic_raw").alias("rare_merged_topic"),
        )
    )

    merged_detail_df = (
        merged_detail_base_df.alias("d")
        .join(
            rare_merged_df.alias("r"),
            on=[
                F.col("d.cate_1_depth") == F.col("r.cate_1_depth"),
                F.col("d.cate_2_depth") == F.col("r.cate_2_depth"),
                F.col("d.sc_measurement") == F.col("r.sc_measurement"),
                F.col("d.merged_topic_raw") == F.col("r.rare_merged_topic"),
            ],
            how="left",
        )
        .withColumn(
            "topic",
            F.when(F.col("r.rare_merged_topic").isNotNull(), F.lit("기타(희소 통합그룹)"))
            .otherwise(F.col("d.merged_topic_raw")),
        )
        .withColumn(
            "description",
            F.when(F.col("r.rare_merged_topic").isNotNull(), F.concat(F.lit("희소 통합그룹 묶음: "), F.col("d.merged_topic_raw")))
            .otherwise(F.col("d.merged_description_raw")),
        )
        .select(
            F.col("d._row_id"),
            F.col("d.cate_1_depth"),
            F.col("d.cate_2_depth"),
            F.col("d.sc_measurement"),
            F.col("d.memo"),
            F.col("d.source_topic"),
            "topic",
            "description",
        )
    )

    save_table(merged_detail_df, MERGED_DETAIL_TABLE)

    merged_summary_df = (
        merged_detail_df
        .groupBy("cate_1_depth", "cate_2_depth", "sc_measurement", "topic")
        .agg(F.count("*").alias("review_count"))
        .withColumn(
            "group_total_count",
            F.sum("review_count").over(
                Window.partitionBy("cate_1_depth", "cate_2_depth", "sc_measurement")
            ),
        )
        .withColumn("review_share", F.round(F.col("review_count") / F.col("group_total_count"), 4))
        .withColumn("review_share_pct", F.round(F.col("review_share") * 100, 2))
        .orderBy("cate_1_depth", "cate_2_depth", "sc_measurement", F.desc("review_count"))
    )

    save_table(merged_summary_df, MERGED_SUMMARY_TABLE)
    display(merged_summary_df)
else:
    print(f"[SKIP] source detail table not found: {SOURCE_DETAIL_TABLE}")


# COMMAND ----------
# 8) Useful Queries

print(f"""
[OUTPUT TABLES]
- mapping:        {TOPIC_MERGE_MAPPING_TABLE}
- merged detail:  {MERGED_DETAIL_TABLE}
- merged summary: {MERGED_SUMMARY_TABLE}

[CHECK EXAMPLE]
select *
from {TOPIC_MERGE_MAPPING_TABLE}
where cate_1_depth = '01. 사이즈'
  and cate_2_depth = '01-01. TV 사이즈'
  and sc_measurement = -1
order by merged_topic, source_topic_order
""")
